# EfficientNetB4 + MS-FSDA: Multi-Scale Frequency-Spatial Dual Attention

**Novel Contribution (updated version):** MS-FSDA + Scale-Consistency + Scale-Wise Temperature
- Extends proven FSDA block to 2 scales (block5 local + block7 semantic)
- **Scale-Consistency Loss:** align attention maps across scales
- **Scale-Wise Temperature:** learnable attention sharpness per scale
- Scale-Adaptive Fusion Gate: content-dependent learnable weighting
- FSDA block identical to proven baseline (93.93% accuracy)

| Component | Detail |
|---|---|
| **Backbone** | EfficientNetB4 (unfreeze blocks 3-7) |
| **Scale 1 (Local)** | block5 output -> FSDA(τ_local) -> GAP |
| **Scale 2 (Semantic)** | block7 output -> FSDA(τ_sem) -> GAP |
| **Fusion** | Scale-Adaptive Gate (softmax weights, content-dependent) |
| **Aux Loss** | Scale-Consistency (cosine) |
| **Loss** | CE (label_smoothing=0.15) + class_weight + λ·Consistency |
| **Metrics** | Acc, P, R, F1, AUC (macro + weighted), Kappa, MCC |


In [ ]:
import os, csv, time, random, shutil, glob, gc
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm_lib
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, Layer, Input, Concatenate, Activation,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger, Callback
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.regularizers import l2

from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score, roc_auc_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

print("=" * 60)
print("  MS-FSDA: Multi-Scale Frequency-Spatial Dual Attention")
print("=" * 60)
print(f"  TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"  GPUs: {len(gpus)}")
else:
    print("  No GPU detected!")
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("  Mixed Precision: mixed_float16")

In [ ]:
STRATEGY_KEY   = "ms_fsda_consistency_temp"
STRATEGY_LABEL = "EfficientNetB4 + MS-FSDA + Consistency + Temp"

DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2944/dataset_final_2944"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

INPUT_SHAPE     = (380, 380, 3)
BATCH_SIZE      = 32
EPOCHS          = 30
LR              = 1e-4
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]
USE_AUG         = True

# FSDA hyperparams (same as proven baseline)
FSDA_REDUCTION  = 16
FSDA_SPATIAL_KS = 7

# NEW: Scale-wise temperature (learnable)
TAU_LOCAL_INIT  = 0.7
TAU_SEM_INIT    = 1.2
TEMP_EPS        = 1e-6

# NEW: Scale-consistency loss
CONSISTENCY_LAMBDA = 0.10

# Head
LABEL_SMOOTHING = 0.15
DROPOUT_RATE    = 0.5
PATIENCE        = 12

# Multi-run
N_RUNS       = 3
RANDOM_SEEDS = [42, 123, 456]
AUTOTUNE     = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)

all_runs_results = []

print(f"  Strategy      : {STRATEGY_LABEL}")
print("  Novel         : 2-Scale FSDA + Scale-Consistency + Scale-Wise Temperature")
print(f"  Loss          : CE (label_smoothing={LABEL_SMOOTHING}) + class_weight + {CONSISTENCY_LAMBDA}×Consistency")
print(f"  FSDA          : reduction={FSDA_REDUCTION}, spatial_ks={FSDA_SPATIAL_KS}")
print(f"  Temperature   : tau_local={TAU_LOCAL_INIT}, tau_sem={TAU_SEM_INIT}")
print(f"  Unfreeze      : {UNFREEZE_BLOCKS}")
print(f"  Runs          : {N_RUNS} × seeds {RANDOM_SEEDS}")

In [ ]:
class FrequencyChannelAttention(tf.keras.layers.Layer):
    """Frequency-domain Channel Attention with learnable temperature.
    FFT2D -> log-magnitude -> channel-wise MLP -> sigmoid gate.
    All computation in float32 for numerical stability.
    """
    def __init__(self, reduction=16, temperature_init=1.0, eps=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction
        self.temperature_init = temperature_init
        self.eps = eps

    def build(self, input_shape):
        C = input_shape[-1]
        r = max(C // self.reduction, 8)
        self.fc1 = Dense(r, use_bias=False, dtype='float32', name=f'{self.name}_fc1')
        self.fc2 = Dense(C, use_bias=False, dtype='float32', name=f'{self.name}_fc2')
        self.fc1.build((None, C))
        self.fc2.build((None, r))
        tau_init = np.log(np.expm1(self.temperature_init))
        self.log_tau = self.add_weight(
            name=f'{self.name}_log_tau', shape=(1,),
            initializer=tf.keras.initializers.Constant(tau_init),
            trainable=True)
        super().build(input_shape)

    def call(self, x, training=False):
        x_f32 = tf.cast(x, tf.float32)
        x_t = tf.transpose(x_f32, [0, 3, 1, 2])
        x_complex = tf.complex(x_t, tf.zeros_like(x_t))
        x_fft = tf.signal.fft2d(x_complex)
        mag = tf.math.log1p(tf.abs(x_fft))
        freq_desc = tf.reduce_mean(mag, axis=[2, 3])
        attn = tf.nn.relu(self.fc1(freq_desc))
        tau = tf.math.softplus(self.log_tau) + self.eps
        attn = tf.nn.sigmoid(self.fc2(attn) / tau)
        attn = tf.reshape(attn, [tf.shape(x_f32)[0], 1, 1, tf.shape(x_f32)[3]])
        out = x_f32 * attn
        return tf.cast(out, x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'reduction': self.reduction,
            'temperature_init': self.temperature_init,
            'eps': self.eps,
        })
        return cfg


class FSDABlock(tf.keras.layers.Layer):
    """Frequency-Spatial Dual Attention (FSDA) Block.
    Returns: (fused_features, spatial_attn_map)
    """
    def __init__(self, reduction=16, spatial_kernel=7,
                 freq_temp_init=1.0, spatial_temp_init=1.0, temp_eps=1e-6, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction
        self.spatial_kernel = spatial_kernel
        self.freq_temp_init = freq_temp_init
        self.spatial_temp_init = spatial_temp_init
        self.temp_eps = temp_eps

    def build(self, input_shape):
        self.freq_attn = FrequencyChannelAttention(
            self.reduction, temperature_init=self.freq_temp_init, eps=self.temp_eps,
            name=f'{self.name}_freq_attn')
        self.sp_conv = Conv2D(
            1, self.spatial_kernel, padding='same', use_bias=False,
            kernel_initializer='glorot_uniform', dtype='float32',
            name=f'{self.name}_sp_conv')
        self.bn = BatchNormalization(dtype='float32', name=f'{self.name}_bn')
        tau_init = np.log(np.expm1(self.spatial_temp_init))
        self.log_tau_spatial = self.add_weight(
            name=f'{self.name}_log_tau_spatial', shape=(1,),
            initializer=tf.keras.initializers.Constant(tau_init),
            trainable=True)
        self.freq_attn.build(input_shape)
        sp_input_shape = tuple(input_shape[:-1]) + (2,)
        self.sp_conv.build(sp_input_shape)
        self.bn.build(input_shape)
        super().build(input_shape)

    def call(self, x, training=False):
        input_dtype = x.dtype
        freq_out = tf.cast(self.freq_attn(x, training=training), tf.float32)
        x_f32 = tf.cast(x, tf.float32)
        avg_pool = tf.reduce_mean(x_f32, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x_f32, axis=-1, keepdims=True)
        sp_logits = self.sp_conv(tf.concat([avg_pool, max_pool], axis=-1))
        tau_spatial = tf.math.softplus(self.log_tau_spatial) + self.temp_eps
        sp_attn = tf.nn.sigmoid(sp_logits / tau_spatial)
        spatial_out = x_f32 * sp_attn
        fused = freq_out + spatial_out
        fused = self.bn(fused, training=training)
        fused = tf.cast(fused, input_dtype)
        return fused, sp_attn

    def compute_output_spec(self, x, training=False):
        import keras
        sp_shape = tuple(x.shape[:-1]) + (1,)
        return (
            keras.KerasTensor(x.shape, dtype=x.dtype),
            keras.KerasTensor(sp_shape, dtype='float32'),
        )

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'reduction': self.reduction,
            'spatial_kernel': self.spatial_kernel,
            'freq_temp_init': self.freq_temp_init,
            'spatial_temp_init': self.spatial_temp_init,
            'temp_eps': self.temp_eps,
        })
        return cfg


class ScaleAdaptiveFusion(tf.keras.layers.Layer):
    """Scale-Adaptive Fusion Gate (Novel contribution).

    Content-dependent learnable weighting of multi-scale features.
    Gate input = concatenation of all scale features.
    Gate output = softmax weights per scale.
    Fused = weighted sum of projected scale features.
    """
    def __init__(self, feat_dim=256, n_scales=2, **kwargs):
        super().__init__(**kwargs)
        self.feat_dim = feat_dim
        self.n_scales = n_scales

    def build(self, input_shape):
        self.projs = []
        total_dim = 0
        for i in range(self.n_scales):
            proj = Dense(self.feat_dim, use_bias=False, dtype='float32',
                         name=f'{self.name}_proj_{i}')
            self.projs.append(proj)
            total_dim += input_shape[i][-1]

        self.gate = Dense(self.n_scales, activation='softmax', dtype='float32',
                          name=f'{self.name}_gate')
        self.bn = BatchNormalization(dtype='float32', name=f'{self.name}_bn')
        super().build(input_shape)

    def call(self, inputs, training=False):
        projected = []
        for i in range(self.n_scales):
            p = self.projs[i](tf.cast(inputs[i], tf.float32))
            projected.append(p)

        gate_input = tf.concat([tf.cast(inp, tf.float32) for inp in inputs], axis=-1)
        scale_weights = self.gate(gate_input)  # (B, n_scales)

        fused = tf.zeros_like(projected[0])
        for i in range(self.n_scales):
            fused = fused + scale_weights[:, i:i+1] * projected[i]

        fused = self.bn(fused, training=training)
        return fused, scale_weights

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'feat_dim': self.feat_dim, 'n_scales': self.n_scales})
        return cfg


CUSTOM_OBJECTS = {
    'FrequencyChannelAttention': FrequencyChannelAttention,
    'FSDABlock': FSDABlock,
    'ScaleAdaptiveFusion': ScaleAdaptiveFusion,
}

print("✅ MS-FSDA Architecture defined")
print("   - FSDA Block: PROVEN (same as baseline 93.93%)")
print("   - NOVEL: Scale-Consistency + Scale-Wise Temperature + 2-Scale Fusion")

In [ ]:
efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def apply_freeze_strategy(base, unfreeze_blocks):
    base.trainable = False
    for layer in base.layers:
        for block_num in unfreeze_blocks:
            if layer.name.startswith(f"block{block_num}"):
                if not isinstance(layer, tf.keras.layers.BatchNormalization):
                    layer.trainable = True
                break
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f"  Backbone: {trainable}/{len(base.layers)} layers trainable (BN frozen)")


def _collect_samples(split_dir, class_to_idx):
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        if not os.path.isdir(d):
            continue
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None, use_aug=True):
    class_names = sorted([d for d in os.listdir(os.path.join(data_dir, 'train'))
                          if os.path.isdir(os.path.join(data_dir, 'train', d))])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes = len(class_names)
    h, w = input_shape[:2]

    def load_and_preprocess(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [h, w])
        img = tf.cast(img, tf.float32)
        img = efficientnet_preprocess(img)
        return img, tf.one_hot(label, depth=num_classes)

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False, apply_aug=False):
        sdir = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if apply_aug:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training).prefetch(AUTOTUNE)
        return ds, len(paths), fns, labels

    train_ds, n_train, _, train_lbl = _make_split('train', training=True, apply_aug=use_aug)
    val_ds, n_val, _, _ = _make_split('val', training=False)
    test_ds, n_test, test_fnames, test_lbl = _make_split('test', training=False)

    cw = class_weight.compute_class_weight('balanced', classes=np.unique(train_lbl), y=train_lbl)
    cw_dict = dict(enumerate(cw))

    meta = SimpleNamespace(
        class_names=class_names, class_to_idx=class_to_idx,
        num_classes=num_classes,
        test_filenames=test_fnames, test_classes=np.array(test_lbl),
        n_train=n_train, n_val=n_val, n_test=n_test,
        class_weight_dict=cw_dict,
    )
    print(f"  Data: train={n_train} val={n_val} test={n_test}")
    print(f"  Classes: {class_names}")
    print(f"  Class weights: {cw_dict}")
    return train_ds, val_ds, test_ds, meta


def build_ms_fsda_model(input_shape, num_classes, steps_per_epoch):
    """Build MS-FSDA: EfficientNetB4 + FSDA on 2 scales + Scale-Adaptive Fusion.

    Novel contribution: Apply proven FSDA block at BOTH block5 (local)
    and block7 (semantic) scales, fuse via learnable gate + consistency loss.
    """
    base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=input_shape)
    apply_freeze_strategy(base, UNFREEZE_BLOCKS)

    # --- Find block5 output for multi-scale ---
    block5_layer = None
    for layer in base.layers:
        if 'block5' in layer.name and 'add' in layer.name:
            block5_layer = layer
    if block5_layer is None:
        for layer in base.layers:
            if 'block5' in layer.name and 'project_bn' in layer.name:
                block5_layer = layer
    if block5_layer is None:
        for layer in base.layers:
            if 'block4' in layer.name and 'add' in layer.name:
                block5_layer = layer

    print(f"  Multi-scale outputs:")
    print(f"    Local   : {block5_layer.name} -> {block5_layer.output.shape}")
    print(f"    Semantic: {base.layers[-1].name} -> {base.output.shape}")

    # --- Build multi-scale backbone ---
    backbone_multi = Model(
        inputs=base.input,
        outputs=[block5_layer.output, base.output],
        name='effnetb4_2scale')

    inputs = Input(shape=input_shape, name='input_image')
    local_map, semantic_map = backbone_multi(inputs)

    # --- FSDA on each scale (PROVEN architecture) ---
    local_attended, local_sp_attn = FSDABlock(
        reduction=FSDA_REDUCTION, spatial_kernel=FSDA_SPATIAL_KS,
        freq_temp_init=TAU_LOCAL_INIT, spatial_temp_init=TAU_LOCAL_INIT,
        temp_eps=TEMP_EPS, name='fsda_local')(local_map)
    semantic_attended, semantic_sp_attn = FSDABlock(
        reduction=FSDA_REDUCTION, spatial_kernel=FSDA_SPATIAL_KS,
        freq_temp_init=TAU_SEM_INIT, spatial_temp_init=TAU_SEM_INIT,
        temp_eps=TEMP_EPS, name='fsda_semantic')(semantic_map)

    # --- GAP each scale ---
    local_feat = GlobalAveragePooling2D(name='gap_local')(local_attended)
    semantic_feat = GlobalAveragePooling2D(name='gap_semantic')(semantic_attended)

    # --- NOVEL: Scale-Adaptive Fusion ---
    fused, scale_weights = ScaleAdaptiveFusion(
        feat_dim=256, n_scales=2, name='scale_fusion')(
        [local_feat, semantic_feat])

    # --- Classification Head (same as proven baseline) ---
    x = BatchNormalization(name='head_bn')(fused)
    x = Dense(256, activation='relu', kernel_regularizer=l2(1e-5), name='head_dense')(x)
    x = Dropout(DROPOUT_RATE, name='head_dropout')(x)
    out = Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)

    model = Model(inputs=inputs, outputs=out, name='MS_FSDA_Model')

    # Visualization model (for spatial attention maps)
    vis_model = Model(inputs=inputs,
                      outputs=[out, local_sp_attn, semantic_sp_attn, scale_weights],
                      name='MS_FSDA_vis')

    # --- NOVEL: Scale-Consistency Loss (attention alignment) ---
    def _resize_to_ref(tensors):
        src, ref = tensors
        ref_size = tf.shape(ref)[1:3]
        return tf.image.resize(src, size=ref_size, method='bilinear')
    local_sp_resized = tf.keras.layers.Lambda(
        _resize_to_ref, name='resize_local_sp')([local_sp_attn, semantic_sp_attn])

    def _consistency_loss(tensors):
        a, b = tensors
        a = tf.reshape(a, [tf.shape(a)[0], -1])
        b = tf.reshape(b, [tf.shape(b)[0], -1])
        a = tf.nn.l2_normalize(a, axis=-1)
        b = tf.nn.l2_normalize(b, axis=-1)
        sim = tf.reduce_mean(tf.reduce_sum(a * b, axis=-1))
        return 1.0 - sim
    consistency = tf.keras.layers.Lambda(
        _consistency_loss, name='scale_consistency')([local_sp_resized, semantic_sp_attn])
    model.add_loss(CONSISTENCY_LAMBDA * consistency)
    model.add_metric(consistency, name='scale_consistency', aggregation='mean')

    # --- Compile with PROVEN loss (same as baseline 93.93%) ---
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR, decay_steps=steps_per_epoch * 5,
        decay_rate=0.9, staircase=True)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=['accuracy'],
    )

    print(f"  Model params: {model.count_params():,}")
    print(f"  Loss: CE (label_smoothing={LABEL_SMOOTHING}) + class_weight + {CONSISTENCY_LAMBDA}×Consistency")
    print(f"  Temp: tau_local={TAU_LOCAL_INIT}, tau_sem={TAU_SEM_INIT}")
    return model, vis_model

print("✅ Model builder + Data pipeline defined")

In [ ]:
for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print("\n" + "=" * 70)
    print(f" RUN {run_idx+1}/{N_RUNS}  seed={seed}  |  {STRATEGY_LABEL}")
    print("=" * 70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed, use_aug=USE_AUG)
    steps_per_epoch = meta.n_train // BATCH_SIZE

    model, vis_model = build_ms_fsda_model(INPUT_SHAPE, meta.num_classes, steps_per_epoch)

    if run_idx == 0:
        print(f"\n  Architecture: MS-FSDA + Consistency + Temp")
        print(f"  FSDA Block: reduction={FSDA_REDUCTION}, spatial_ks={FSDA_SPATIAL_KS}")
        print(f"  Consistency λ: {CONSISTENCY_LAMBDA}")
        print(f"  Temperature : tau_local={TAU_LOCAL_INIT}, tau_sem={TAU_SEM_INIT}")
        print(f"  Classifier: BN→Dense(256)→Dropout({DROPOUT_RATE})→Softmax")

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv'), append=False),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'best_model.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ]

    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS, class_weight=meta.class_weight_dict,
        callbacks=callbacks)

    # Learning curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Val')
    axes[0].set_title(f'Accuracy — Run {run_idx+1}'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Val')
    axes[1].set_title(f'Loss — Run {run_idx+1}'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curve.png'), dpi=300)
    plt.show()

    # Evaluate on test
    best_model = load_model(
        os.path.join(RESULT_DIR, 'best_model.keras'),
        custom_objects=CUSTOM_OBJECTS)
    pred_probs = best_model.predict(test_ds, verbose=0)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes
    class_names = meta.class_names

    report = classification_report(
        y_true_run, y_pred_run,
        target_names=class_names, output_dict=True, digits=4)
    test_acc = np.mean(y_pred_run == y_true_run)

    # AUC
    nc = meta.num_classes
    y_bin = label_binarize(y_true_run, classes=list(range(nc)))
    try:
        macro_auc = roc_auc_score(y_bin, pred_probs, multi_class='ovr', average='macro')
        weighted_auc = roc_auc_score(y_bin, pred_probs, multi_class='ovr', average='weighted')
    except:
        macro_auc = weighted_auc = 0.0

    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(y_true_run, y_pred_run,
                                      target_names=class_names, digits=4))
        f.write(f"\nMacro AUC: {macro_auc:.4f}\nWeighted AUC: {weighted_auc:.4f}\n")

    # Confusion matrix
    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names,
                yticklabels=class_names, cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'CM — Run {run_idx+1} (Acc={test_acc:.4f}, AUC={macro_auc:.4f})')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.show(); plt.close()

    all_runs_results.append({
        'run': run_idx+1, 'seed': seed,
        'accuracy': test_acc,
        'precision': report['weighted avg']['precision'],
        'recall': report['weighted avg']['recall'],
        'f1_score': report['weighted avg']['f1-score'],
        'macro_auc': macro_auc,
        'weighted_auc': weighted_auc,
        'per_class_metrics': {c: report[c] for c in class_names},
        'result_dir': RESULT_DIR,
        'history': history.history,
        'y_true': y_true_run, 'y_pred': y_pred_run,
        'pred_probs': pred_probs,
        'class_names': class_names,
        'test_filenames': meta.test_filenames,
        'n_train': meta.n_train, 'n_val': meta.n_val, 'n_test': meta.n_test,
        'best_model_path': os.path.join(RESULT_DIR, 'best_model.keras'),
    })

    print(f"\n  ✅ Run {run_idx+1}: Acc={test_acc:.4f}  P={report['weighted avg']['precision']:.4f}  "
          f"R={report['weighted avg']['recall']:.4f}  F1={report['weighted avg']['f1-score']:.4f}")
    print(f"     AUC(macro)={macro_auc:.4f}  AUC(weighted)={weighted_auc:.4f}")

    tf.keras.backend.clear_session()
    gc.collect()

# Save summary CSV
summary_path = os.path.join(BASE_RESULT_DIR, 'strategy_summary.csv')
with open(summary_path, 'w', newline='') as csvf:
    fieldnames = ['strategy_key', 'strategy_label', 'run', 'seed',
                  'accuracy', 'precision', 'recall', 'f1_score',
                  'macro_auc', 'weighted_auc']
    writer = csv.DictWriter(csvf, fieldnames=fieldnames)
    writer.writeheader()
    for r in all_runs_results:
        writer.writerow({
            'strategy_key': STRATEGY_KEY, 'strategy_label': STRATEGY_LABEL,
            'run': r['run'], 'seed': r['seed'],
            'accuracy': r['accuracy'], 'precision': r['precision'],
            'recall': r['recall'], 'f1_score': r['f1_score'],
            'macro_auc': r['macro_auc'], 'weighted_auc': r['weighted_auc'],
        })

print("\n" + "=" * 70)
print(f" ALL {N_RUNS} RUNS COMPLETED — {STRATEGY_LABEL}")
print("=" * 70)

In [ ]:
accuracies  = [r['accuracy'] for r in all_runs_results]
precisions  = [r['precision'] for r in all_runs_results]
recalls     = [r['recall'] for r in all_runs_results]
f1_scores   = [r['f1_score'] for r in all_runs_results]
macro_aucs  = [r['macro_auc'] for r in all_runs_results]
weighted_aucs = [r['weighted_auc'] for r in all_runs_results]
class_names = all_runs_results[0]['class_names']

print(f"\n{'=' * 60}")
print(f"  {STRATEGY_LABEL}")
print(f"{'=' * 60}")
print(f"  Accuracy     : {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  Precision    : {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")
print(f"  Recall       : {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
print(f"  F1-Score     : {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  Macro AUC    : {np.mean(macro_aucs):.4f} ± {np.std(macro_aucs):.4f}")
print(f"  Weighted AUC : {np.mean(weighted_aucs):.4f} ± {np.std(weighted_aucs):.4f}")
print(f"  Per run acc  : {[f'{a:.4f}' for a in accuracies]}")

print("\n  ADDITIONAL METRICS:")
for r in all_runs_results:
    kappa = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    print(f"    Run {r['run']}: BalAcc={bal_acc:.4f}  Kappa={kappa:.4f}  MCC={mcc:.4f}")

print("\n  PER-CLASS METRICS (mean ± std):")
per_class_stats = {}
for cn in class_names:
    p_vals = [r['per_class_metrics'][cn]['precision'] for r in all_runs_results]
    r_vals = [r['per_class_metrics'][cn]['recall'] for r in all_runs_results]
    f_vals = [r['per_class_metrics'][cn]['f1-score'] for r in all_runs_results]
    per_class_stats[cn] = {
        'precision': {'mean': np.mean(p_vals), 'std': np.std(p_vals)},
        'recall': {'mean': np.mean(r_vals), 'std': np.std(r_vals)},
        'f1': {'mean': np.mean(f_vals), 'std': np.std(f_vals)},
    }
    print(f"    {cn:<28} P={np.mean(p_vals):.4f}±{np.std(p_vals):.4f}  "
          f"R={np.mean(r_vals):.4f}±{np.std(r_vals):.4f}  "
          f"F1={np.mean(f_vals):.4f}±{np.std(f_vals):.4f}")

# Save CSVs
overall_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Macro_AUC', 'Weighted_AUC'],
    'Mean': [np.mean(v) for v in [accuracies, precisions, recalls, f1_scores, macro_aucs, weighted_aucs]],
    'Std': [np.std(v) for v in [accuracies, precisions, recalls, f1_scores, macro_aucs, weighted_aucs]],
})
overall_df.to_csv(os.path.join(BASE_RESULT_DIR, 'overall_metrics_summary.csv'), index=False)
print("\n✅ Results aggregation complete")

In [ ]:
# --- 7.1: Per-Class Metrics Bar Chart ---
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(class_names))
width = 0.25
colors = ['#2196F3', '#4CAF50', '#FF9800']
for i, (metric, label) in enumerate([('precision','Precision'), ('recall','Recall'), ('f1','F1-Score')]):
    means = [per_class_stats[cn][metric]['mean'] for cn in class_names]
    stds = [per_class_stats[cn][metric]['std'] for cn in class_names]
    ax.bar(x + i*width, means, width, yerr=stds, label=label,
           color=colors[i], alpha=0.85, capsize=4, edgecolor='white')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title(f'Per-Class Metrics — {STRATEGY_LABEL}\n(Mean ± Std over {N_RUNS} runs)', fontweight='bold')
ax.set_xticks(x + width); ax.set_xticklabels(class_names, rotation=20, ha='right')
ax.set_ylim(0, 1.05); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'per_class_metrics.png'), dpi=300)
plt.show()

# --- 7.2: Aggregate Confusion Matrix (Raw + Normalized) ---
agg_cm = np.zeros((len(class_names), len(class_names)), dtype=int)
for r in all_runs_results:
    agg_cm += confusion_matrix(r['y_true'], r['y_pred'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(agg_cm, annot=True, fmt='d', xticklabels=class_names,
            yticklabels=class_names, cmap='Blues', ax=axes[0])
axes[0].set_title(f'Aggregate CM (raw, {N_RUNS} runs)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

cm_norm = agg_cm.astype(float) / agg_cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.3f', xticklabels=class_names,
            yticklabels=class_names, cmap='YlOrRd', ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Aggregate CM (normalized)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'agg_confusion_matrix.png'), dpi=300)
plt.show()

# --- 7.3: ROC Curves + AUC ---
best_run = max(all_runs_results, key=lambda r: r['accuracy'])
nc = len(class_names)
y_bin = label_binarize(best_run['y_true'], classes=list(range(nc)))
pred_probs = best_run['pred_probs']

fig, ax = plt.subplots(figsize=(8, 7))
colors_roc = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12', '#9b59b6']
for i, cn in enumerate(class_names):
    fpr_i, tpr_i, _ = roc_curve(y_bin[:, i], pred_probs[:, i])
    roc_auc_i = auc(fpr_i, tpr_i)
    ax.plot(fpr_i, tpr_i, color=colors_roc[i % len(colors_roc)],
            lw=2, label=f'{cn} (AUC={roc_auc_i:.4f})')

fpr_m, tpr_m, _ = roc_curve(y_bin.ravel(), pred_probs.ravel())
auc_m = auc(fpr_m, tpr_m)
ax.plot(fpr_m, tpr_m, 'k--', lw=2, label=f'Macro avg (AUC={auc_m:.4f})')
ax.plot([0, 1], [0, 1], 'gray', ls=':', lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — {STRATEGY_LABEL}\n(Best run: seed={best_run["seed"]})', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'roc_curves.png'), dpi=300)
plt.show()
print("✅ All thesis visualizations saved")

In [ ]:
gc.collect()
print("Computing t-SNE embeddings...")

best_model_path = best_run['best_model_path']
if os.path.exists(best_model_path):
    feat_model = load_model(best_model_path, custom_objects=CUSTOM_OBJECTS)
    try:
        feat_extractor = Model(inputs=feat_model.input,
                               outputs=feat_model.get_layer('head_dense').output)
    except:
        feat_extractor = Model(inputs=feat_model.input,
                               outputs=feat_model.layers[-3].output)

    random.seed(best_run['seed']); np.random.seed(best_run['seed']); tf.random.set_seed(best_run['seed'])
    _, _, test_ds_tsne, meta_tsne = create_tf_datasets(DATA_DIR, INPUT_SHAPE, 4, seed=best_run['seed'], use_aug=False)

    features = feat_extractor.predict(test_ds_tsne, verbose=1)
    y_true_tsne = meta_tsne.test_classes

    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(y_true_tsne)-1))
    features_2d = tsne.fit_transform(features)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors_tsne = ['#e74c3c', '#2ecc71', '#3498db']
    for i, cn in enumerate(meta_tsne.class_names):
        mask = y_true_tsne == i
        ax.scatter(features_2d[mask, 0], features_2d[mask, 1],
                   c=colors_tsne[i % len(colors_tsne)], label=cn,
                   alpha=0.7, s=40, edgecolors='white', lw=0.5)
    ax.set_title(f't-SNE Feature Visualization — {STRATEGY_LABEL}', fontweight='bold', fontsize=13)
    ax.legend(fontsize=10, markerscale=1.5); ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_RESULT_DIR, 'tsne.png'), dpi=300)
    plt.show()
    print("✅ t-SNE saved")
    del feat_model, feat_extractor, features
    tf.keras.backend.clear_session(); gc.collect()
else:
    print("⚠️ Best model not found, skipping t-SNE")

In [ ]:
report_lines = [
    "=" * 80,
    "MS-FSDA: MULTI-SCALE FREQUENCY-SPATIAL DUAL ATTENTION — EXPERIMENT REPORT",
    "=" * 80,
    f"Strategy: {STRATEGY_LABEL}",
    "",
    "NOVEL CONTRIBUTION:",
    "  Multi-Scale FSDA (MS-FSDA)",
    "  - Extends proven FSDA block to operate on 2 scales of EfficientNetB4:",
    "    Scale 1 (Local):    block5 output — captures local textures, small defects",
    "    Scale 2 (Semantic): block7 output — captures global shape, color distribution",
    "  - Scale-Adaptive Fusion Gate: content-dependent learnable weighting",
    "  - Scale-Consistency Loss: align attention maps across scales",
    "  - Scale-Wise Temperature: learnable attention sharpness per scale",
    "  - FSDA block unchanged from baseline (proven 93.93%)",
    "",
    f"Architecture: EfficientNetB4 + 2×FSDA + Fusion + Consistency + Temp",
    f"  FSDA: reduction={FSDA_REDUCTION}, spatial_ks={FSDA_SPATIAL_KS}",
    f"  Consistency λ: {CONSISTENCY_LAMBDA}",
    f"  Temperature: tau_local={TAU_LOCAL_INIT}, tau_sem={TAU_SEM_INIT}",
    f"  Unfreeze blocks: {UNFREEZE_BLOCKS}",
    f"  Loss: CE (label_smoothing={LABEL_SMOOTHING}) + class_weight + λ·Consistency",
    f"  LR: {LR} (ExponentialDecay)",
    "",
    f"Dataset: {DATA_DIR}",
    f"Runs: {N_RUNS} seeds = {RANDOM_SEEDS[:N_RUNS]}",
    "",
    "OVERALL PERFORMANCE (Mean ± Std):",
    "-" * 60,
    f"  Accuracy     : {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}",
    f"  Precision    : {np.mean(precisions):.4f} ± {np.std(precisions):.4f}",
    f"  Recall       : {np.mean(recalls):.4f} ± {np.std(recalls):.4f}",
    f"  F1-Score     : {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}",
    f"  Macro AUC    : {np.mean(macro_aucs):.4f} ± {np.std(macro_aucs):.4f}",
    f"  Weighted AUC : {np.mean(weighted_aucs):.4f} ± {np.std(weighted_aucs):.4f}",
    "",
    "PER-CLASS F1-SCORE:",
]
for cn in class_names:
    s = per_class_stats[cn]['f1']
    report_lines.append(f"  {cn:<30} {s['mean']:.4f} ± {s['std']:.4f}")
report_lines.extend(["", "ADDITIONAL METRICS:"])
for r in all_runs_results:
    kappa = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc = matthews_corrcoef(r['y_true'], r['y_pred'])
    bal_acc = balanced_accuracy_score(r['y_true'], r['y_pred'])
    report_lines.append(f"  Run {r['run']}: BalAcc={bal_acc:.4f}  Kappa={kappa:.4f}  MCC={mcc:.4f}  "
                        f"AUC(macro)={r['macro_auc']:.4f}  AUC(weighted)={r['weighted_auc']:.4f}")

report_text = "\n".join(report_lines)
print(report_text)
with open(os.path.join(BASE_RESULT_DIR, 'EXPERIMENT_REPORT.txt'), 'w', encoding='utf-8') as f:
    f.write(report_text)

# ZIP
zip_path = f"/kaggle/working/{STRATEGY_KEY}_complete"
shutil.make_archive(zip_path, 'zip', BASE_RESULT_DIR)
zip_size = os.path.getsize(f"{zip_path}.zip") / (1024*1024)
print(f"\n✅ Archive: {zip_path}.zip ({zip_size:.2f} MB)")
print("DONE! 🎉")